# Semaine 9 — Jour 4 : Hybrid Search : BM25 et fusion des rangs

Calculer un classement lexical BM25 sur le corpus autorisé.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. Les modèles et services optionnels ajoutent un temps d’installation distinct.

Prérequis : Python 3.11+, JSON, notions de vecteurs et de persistance. Le parcours obligatoire utilise uniquement la bibliothèque standard et ne contacte aucun service externe.

Lire [les objectifs](learning_objectives.md), [le chapitre](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week09/S9_J4_hybrid_search.ipynb` et `notebooks/week09/teacher/S9_J4_hybrid_search_teacher.ipynb`.

Critère de réussite : Mesurer la couverture des candidats avant de choisir un top-k.

Les calculs de retrieval et de persistance sont exécutables; le corpus est fictif. L’encodeur de base est lexical et la réponse finale est extractive. Les extensions neuronales sont séparées et n’ont pas été exécutées ici.


# Objectifs mesurables

- Calculer un classement lexical BM25 sur le corpus autorisé.
- Fusionner deux classements par Reciprocal Rank Fusion sans additionner des scores incompatibles.
- Mesurer la couverture des candidats avant de choisir un top-k.

## Preuves attendues

Une sortie reproductible, un contrôle d’invariant et une limite du mécanisme correctement expliquée.


# S9 J4 — Hybrid Search : BM25 et fusion des rangs


## 1. Pourquoi deux canaux ?
Une recherche lexicale valorise les termes présents, notamment codes d’erreur, identifiants et noms de composants. Un encodeur neuronal peut retrouver des paraphrases. La recherche hybride vise une complémentarité, pas un remplacement automatique du lexical. Dans ce lab, les deux canaux restent lexicaux : BM25 et projection hashbow. Leur fusion démontre le protocole, sans prouver un gain sémantique.

La requête q8 utilise un vocabulaire éloigné des passages pertinents. Elle rend visible le défaut commun des deux représentations. Si deux canaux partagent la même faiblesse, fusionner leurs rangs ne crée pas l’information manquante.

## 2. BM25 local
Le score dépend de la fréquence d’un terme dans un passage, de sa rareté dans le corpus et de la longueur du passage. La saturation évite qu’une répétition illimitée du même mot augmente linéairement le score. La normalisation de longueur évite de favoriser systématiquement les grands passages.

Le module utilise k1=1,2, b=0,75 et une formule IDF positive explicite dans le code. Les termes de la requête sont dédupliqués. La tokenisation est celle du lab; elle n’a pas de stemming ni de compréhension des synonymes. Les paramètres sont une baseline, pas une recommandation universelle.

## 3. Des scores non comparables
Un score BM25 n’a pas la même distribution qu’un cosinus. Les additionner sans calibration attribue une importance arbitraire aux deux systèmes. La fusion RRF utilise les rangs : pour un candidat, additionner 1/(c + rang) dans les listes où il apparaît. Les rangs commencent à 1 et c vaut 60 dans le lab.

Un candidat absent d’une liste ne reçoit aucune contribution de celle-ci. Un même chunk ne peut pas apparaître deux fois dans une seule liste. La fusion conserve les métadonnées et trie les égalités de manière déterministe par identifiant. La constante amortit les écarts de rang; elle n’est ni un seuil de confiance ni une probabilité.

## 4. Profondeur de candidats
candidate_k borne le nombre de passages de chaque canal avant fusion. top_k borne le nombre final retourné. Si un passage manque dans toutes les listes de candidats, la fusion et le reranker ne pourront pas le récupérer. Évaluer le rappel à une profondeur suffisamment grande avant d’optimiser seulement le premier résultat.

Les métriques de la semaine sont calculées au niveau document. Plusieurs chunks d’une même source sont dédupliqués en gardant la première occurrence avant Recall@k, MRR@k et nDCG@k. Sans cette convention, un document possédant de nombreux chunks pourrait occuper tous les rangs et masquer l’absence d’autres sources.

## 5. Filtrage et protocole
Les deux canaux voient les mêmes passages autorisés. BM25 calcule ses fréquences documentaires sur ce sous-corpus. Le vecteur de requête est comparé uniquement aux vecteurs de ce périmètre. Le corpus, la tokenisation et la dimension sont tracés dans les rapports. Un changement du corpus demande une nouvelle mesure, comme les changements de protocole de S8.

Le petit jeu de développement ne doit pas devenir une preuve de production. Les questions proches du texte peuvent être faciles pour toutes les variantes. Examiner q8 et ajouter de nouveaux cas annotés sans transformer silencieusement les qrels pour améliorer le résultat.

## 6. Expérience du jour
Construire deux listes simples, calculer RRF à la main et vérifier le résultat du code. Comparer lexical, vector et hybrid sur toutes les requêtes. Une égalité de performance est un résultat acceptable; le cours ne doit pas annoncer une supériorité qui n’est pas observée.

## À retenir
L’hybridation combine des preuves de retrieval avec un protocole explicite. Elle ne rend pas compatibles deux espaces par magie et ne répare pas un candidat absent des listes.


## Diagramme du jour

```mermaid
flowchart TD
 A["Passages autorisés"] --> B["BM25"]
 A --> C["Similarité vectorielle"]
 B --> D["Rangs de candidats"]
 C --> D
 D --> E["Fusion RRF"]
 E --> F["Top-k et mesure du rappel"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

Les cellules s’exécutent hors ligne depuis l’arbre du dépôt.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week09/knowledge.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
import math
from examples.week09.knowledge import rrf, ContractError
A = {"chunk_id": "A", "score": 20}
B = {"chunk_id": "B", "score": .9}
C = {"chunk_id": "C", "score": .1}
result = rrf([[A, B, C], [B, A]])
scores = {c["chunk_id"]: c["score"] for c in result}
assert math.isclose(scores["A"], 1/61 + 1/62)
assert scores["A"] == scores["B"]
assert math.isclose(scores["C"], 1/63)
print(result)

# Lab J4 — Hybrid Search : BM25 et fusion des rangs

Depuis la racine du dépôt :

```bash
python3 book/week09/day04/labs/run.py
python3 -m examples.week09.demo --day 4
```

Résultat attendu : **classements lexical, vector et hybrid inspectables**.

## Manipulations

1. Reproduire la sortie et retrouver le calcul dans `examples/week09/knowledge.py`.
2. Réaliser l’exercice 2 avec une copie des données, sans modifier le corpus distribué.
3. Vérifier l’invariant étudié et expliquer une limite.

Le corpus se trouve dans `examples/week09/corpus`. Les tests automatisés sont sous `tests/week09`. [QUICKSTART](../../../../docs/week09/QUICKSTART.md) décrit les commandes et les codes retour.

Les modèles locaux et pgvector sont optionnels : lire `examples/week09/optional/README.md` depuis la racine du dépôt. Ils ne sont pas nécessaires pour finir cette journée et ne sont pas présentés comme validés dans cet environnement.


# Exercices

Conserver votre première réponse avant de lire les solutions.

## Exercice 1 — Calculer RRF

Avec c=60, A est rang 1 dans la première liste et rang 2 dans la seconde. Calculer son score.

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 2 — Détecter une fausse fusion

Pourquoi BM25 + cosinus sans normalisation est-il difficile à interpréter ?

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 3 — Comprendre la profondeur

Un passage pertinent est rang 15 dans les deux canaux avec candidate_k=12. Le reranker peut-il le récupérer ?

Livrable : réponse argumentée avec calcul, test ou observation.


# Entretien

Répondre en deux minutes avec un exemple concret et une limite.

## Question 1

Quand l’hybride n’améliore-t-il rien ?

## Question 2

Pourquoi évaluer au niveau document ?


# Challenge

Construire deux classements inversés de A/B, vérifier leur égalité de score RRF, puis ajouter C seulement dans une liste. Vérifier qu’il n’obtient qu’une contribution.

## Critères de réussite

Reproductibilité, provenance des données, invariant vérifié et limite expliquée sans exagérer les garanties.


# Corrigé des exercices

## Exercice 1

1/61 + 1/62, environ 0,03252. Le score vient de la position, pas de la magnitude des deux scores bruts.

## Exercice 2

Les échelles et distributions diffèrent. L’addition donne une pondération implicite instable. RRF évite cette calibration en utilisant les rangs, avec ses propres limites.

## Exercice 3

Non. Il est absent des candidats. Augmenter ou diversifier le retrieval, puis mesurer l’effet sur le coût et le rappel.


# Réponses d’entretien

## Question 1

Quand les canaux ont les mêmes erreurs ou que les candidats pertinents sont absents des deux. L’hybridation doit être mesurée sur la tâche.

## Question 2

Pour ne pas compter plusieurs chunks de la même source comme plusieurs documents retrouvés. Une évaluation au passage est aussi possible avec des qrels adaptés.


# Solution du challenge

RRF([A,B],[B,A]) attribue la même somme à A et B; l’identifiant départage le tri. C au rang 3 d’une seule liste vaut 1/(60+3). Injecter deux fois le même identifiant dans une liste doit produire ContractError.

## Vérification exécutable

Exécuter après les cellules du lab élève.

In [ ]:
try:
    rrf([[A, A]])
except ContractError:
    print("Double contribution dans une même liste bloquée")
else:
    raise AssertionError("Doublon accepté")
from examples.week09.demo import populated, evaluate_retrieval
store = populated()
try:
    print(evaluate_retrieval(store)["summaries"])
finally:
    store.close()

# Revue pédagogique — Jour 4

## Acquis

- Calculer un classement lexical BM25 sur le corpus autorisé.
- Fusionner deux classements par Reciprocal Rank Fusion sans additionner des scores incompatibles.
- Mesurer la couverture des candidats avant de choisir un top-k.

## Barème indicatif sur 10

- 3 : concepts correctement reliés au problème.
- 3 : lab reproduit et résultat interprété.
- 2 : exercices argumentés.
- 2 : challenge et limite identifiée.

Le parcours essentiel peut être terminé sans le challenge. Corriger toute confusion entre hashbow et modèle neuronal, entre tenant fourni et authentification, ou entre citation valide et vérité du document. Vérifier en particulier : Non. Il est absent des candidats. Augmenter ou diversifier le retrieval, puis mesurer l’effet sur le coût et le rappel.


# Références — J4

Sources primaires consultées le 20 octobre 2026. Les algorithmes et contrats du lab sont explicités dans le chapitre; le corpus est une création fictive du cours.

- [pgvector : documentation officielle](https://github.com/pgvector/pgvector) — stockage, distances, recherche exacte/ANN et filtres.
- [Sentence Transformers : retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) — architecture de récupération en deux étapes.
- [SentenceTransformer : API](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html) — encode_query, encode_document et modèles locaux.
- [CrossEncoder : API](https://www.sbert.net/docs/package_reference/cross_encoder/model.html) — scoring conjoint de paires.
- [Cormack, Clarke et Buettcher, SIGIR 2009 : Reciprocal Rank Fusion](https://research.google/pubs/reciprocal-rank-fusion-outperforms-condorcet-and-individual-rank-learning-methods/) — origine de la méthode de fusion des rangs.
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) — persistance locale et transactions.

Lire en priorité les références du thème du jour. Les extensions évoluent; leur environnement réel doit être verrouillé et testé avant utilisation. Le paquet n’affirme pas avoir exécuté de modèle neuronal ou de serveur pgvector.
